In [ ]:
!pip install -q -U transformers accelerate sentencepiece safetensors
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
!cp -r /kaggle/input/datasets/rifatbinreza/alta2026-pipeline/alta2026_pipeline /kaggle/working/
%cd /kaggle/working/alta2026_pipeline
!ls

In [ ]:
!python classical_baseline.py --train train.csv --valid valid.csv

In [ ]:
%%writefile /kaggle/working/alta2026_pipeline/transformer_multitask.py
from __future__ import annotations

import argparse
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

from metrics import alta_score, competition_stratify_key
from thresholds import optimize_thresholds, apply_thresholds, save_thresholds

try:
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError as e:
    raise SystemExit(
        "transformers is required. Install with: pip install -r requirements.txt"
    ) from e


@dataclass
class Config:
    model_name: str = "microsoft/deberta-v3-large"
    max_length: int = 384
    batch_size: int = 8
    epochs: int = 4
    lr: float = 1.5e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    dropout: float = 0.15
    sentiment_loss_weight: float = 1.0
    sarcasm_loss_weight: float = 1.25
    sarcasm_pos_weight: float = 1.0
    gradient_accumulation_steps: int = 4
    fp16: bool = False
    seed: int = 42
    n_folds: int = 5
    early_stopping_patience: int = 2


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


SPECIAL_TOKENS = ["[SRC_GOOGLE]", "[SRC_REDDIT]", "[VAR_EN_AU]", "[VAR_EN_UK]"]

def add_prefix(df: pd.DataFrame) -> pd.Series:
    return (
        "[SRC_" + df.source.str.upper() + "] [VAR_" + df.variety.str.replace("-", "_").str.upper() + "] "
        + df.text.fillna("")
    )


class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.texts = add_prefix(self.df).tolist()
        self.max_length = max_length
        self.with_labels = with_labels

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i], max_length=self.max_length,
            truncation=True, padding="max_length", return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.with_labels:
            item["sentiment"] = torch.tensor(int(self.df.loc[i, "sentiment"]), dtype=torch.long)
            item["sarcasm"] = torch.tensor(int(self.df.loc[i, "sarcasm"]), dtype=torch.long)
        return item


class MultiTaskModel(nn.Module):
    def __init__(self, name: str, dropout: float):
        super().__init__()
        # Force float32: some hub checkpoints default-load in fp16, which
        # then mismatches with our plain-float32 classifier heads.
        self.encoder = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        h = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.sent_head = nn.Linear(h, 2)
        self.sarc_head = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**kwargs)
        pooled = out.last_hidden_state[:, 0]
        pooled = self.drop(pooled)
        return self.sent_head(pooled), self.sarc_head(pooled)


def evaluate(model, loader, device):
    model.eval(); ps, pz = [], []; ys, yz = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            ps.append(torch.softmax(a, -1)[:, 1].cpu().numpy())
            pz.append(torch.softmax(b, -1)[:, 1].cpu().numpy())
            ys.append(batch["sentiment"].numpy()); yz.append(batch["sarcasm"].numpy())
    return np.concatenate(ps), np.concatenate(pz), np.concatenate(ys), np.concatenate(yz)


def train_one_fold(train_df, val_df, cfg, out_dir, fold):
    set_seed(cfg.seed + fold)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    model = MultiTaskModel(cfg.model_name, cfg.dropout)
    model.encoder.resize_token_embeddings(len(tokenizer))
    model = model.to(device)

    tr_ds = TextDataset(train_df, tokenizer, cfg.max_length, True)
    va_ds = TextDataset(val_df, tokenizer, cfg.max_length, True)
    tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
    va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_loader) * cfg.epochs // cfg.gradient_accumulation_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(steps * cfg.warmup_ratio), steps
    )

    s_loss = nn.CrossEntropyLoss()
    z_weight = torch.tensor([1.0, cfg.sarcasm_pos_weight], device=device)
    z_loss = nn.CrossEntropyLoss(weight=z_weight)
    best = -1.0
    best_state = None
    epochs_since_improve = 0

    for epoch in range(cfg.epochs):
        model.train(); optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            ids = batch["input_ids"].to(device); mask = batch["attention_mask"].to(device)
            tt = batch.get("token_type_ids")
            if tt is not None: tt = tt.to(device)
            a, b = model(ids, mask, tt)
            loss = (cfg.sentiment_loss_weight * s_loss(a, batch["sentiment"].to(device))
                    + cfg.sarcasm_loss_weight * z_loss(b, batch["sarcasm"].to(device)))
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            if (step + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)

        ps, pz, ys, yz = evaluate(model, va_loader, device)
        tmp = val_df[["source", "variety", "text"]].copy()
        tmp["sentiment"] = (ps >= 0.5).astype(int)
        tmp["sarcasm"] = (pz >= 0.5).astype(int)
        scores, comp = alta_score(val_df, tmp)
        print(f"fold={fold} epoch={epoch+1} score={comp:.5f} {scores}")
        if comp > best:
            best = comp
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= cfg.early_stopping_patience:
                print(f"fold={fold}: no improvement for {cfg.early_stopping_patience} epochs, stopping early at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), out_dir / f"fold{fold}.pt")
    meta = {"fold": fold, "best_validation_score": best, "config": asdict(cfg)}
    with open(out_dir / f"fold{fold}.json", "w") as f: json.dump(meta, f, indent=2)
    return model, tokenizer, device


def cv_train(train_csv: str, out_dir: str, cfg: Config, external_valid_csv: str | None = None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(train_csv)
    keys = competition_stratify_key(df)
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)
    oof = df[["source", "variety", "text", "sentiment", "sarcasm"]].copy()
    oof["p_sentiment"] = np.nan; oof["p_sarcasm"] = np.nan

    ext = pd.read_csv(external_valid_csv) if external_valid_csv else None
    ext_ps_all, ext_pz_all = [], []

    for fold, (tr, va) in enumerate(skf.split(df, keys)):
        model, tokenizer, device = train_one_fold(df.iloc[tr].copy(), df.iloc[va].copy(), cfg, out, fold)
        va_ds = TextDataset(df.iloc[va].copy(), tokenizer, cfg.max_length, True)
        va_loader = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False)
        ps, pz, _, _ = evaluate(model, va_loader, device)
        oof.loc[df.index[va], "p_sentiment"] = ps
        oof.loc[df.index[va], "p_sarcasm"] = pz

        if ext is not None:
            ext_ds = TextDataset(ext.copy(), tokenizer, cfg.max_length, False)
            ext_loader = DataLoader(ext_ds, batch_size=cfg.batch_size * 2, shuffle=False)
            model.eval(); eps=[]; epz=[]
            with torch.no_grad():
                for batch in ext_loader:
                    ids=batch["input_ids"].to(device); mask=batch["attention_mask"].to(device)
                    tt=batch.get("token_type_ids")
                    if tt is not None: tt=tt.to(device)
                    a,b=model(ids,mask,tt)
                    eps.append(torch.softmax(a,-1)[:,1].cpu().numpy())
                    epz.append(torch.softmax(b,-1)[:,1].cpu().numpy())
            ext_ps_all.append(np.concatenate(eps)); ext_pz_all.append(np.concatenate(epz))

    oof.to_csv(out / "oof_probabilities.csv", index=False)
    pred = oof[["source", "variety", "text"]].copy()
    pred["sentiment"] = (oof.p_sentiment >= .5).astype(int)
    pred["sarcasm"] = (oof.p_sarcasm >= .5).astype(int)
    scores, comp = alta_score(df, pred)
    print("OOF @0.5", scores, comp)

    if ext is not None:
        ext_probs = ext[["source", "variety", "text"]].copy()
        ext_probs["p_sentiment"] = np.mean(ext_ps_all, axis=0)
        ext_probs["p_sarcasm"] = np.mean(ext_pz_all, axis=0)
        ext_probs.to_csv(out / "valid_probabilities.csv", index=False)
        ext_pred = ext_probs[["source", "variety", "text"]].copy()
        ext_pred["sentiment"] = (ext_probs.p_sentiment >= .5).astype(int)
        ext_pred["sarcasm"] = (ext_probs.p_sarcasm >= .5).astype(int)
        ext_scores, ext_comp = alta_score(ext, ext_pred)
        print("External valid @0.5", ext_scores, ext_comp)

def calibrate(valid_csv: str, probs_csv: str, out_dir: str):
    valid = pd.read_csv(valid_csv)
    probs = pd.read_csv(probs_csv)
    th = optimize_thresholds(valid, probs)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    save_thresholds(th, os.path.join(out_dir, "thresholds.json"))
    pred = apply_thresholds(probs, th)
    scores, comp = alta_score(valid, pred)
    pred.to_csv(os.path.join(out_dir, "calibrated_valid_predictions.csv"), index=False)
    print(json.dumps({"scores": scores, "alta_score": comp, "thresholds": th}, indent=2))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["cv"], default="cv")
    ap.add_argument("--train", default="../train.csv")
    ap.add_argument("--valid", default=None)
    ap.add_argument("--out", default="artifacts/transformer")
    ap.add_argument("--model", default="microsoft/deberta-v3-large")
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--max-length", type=int, default=384)
    ap.add_argument("--n-folds", type=int, default=5)
    ap.add_argument("--patience", type=int, default=2, help="stop a fold early after this many epochs with no score improvement")
    ap.add_argument("--grad-accum", type=int, default=4, help="gradient accumulation steps; raise this and lower --batch-size together if you hit CUDA out-of-memory")
    args, _unknown = ap.parse_known_args()
    cfg = Config(model_name=args.model, epochs=args.epochs, batch_size=args.batch_size,
                 max_length=args.max_length, n_folds=args.n_folds, early_stopping_patience=args.patience,
                 gradient_accumulation_steps=args.grad_accum)
    cv_train(args.train, args.out, cfg, args.valid)

In [ ]:
!python transformer_multitask.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/transformer_base \
  --model microsoft/deberta-v3-base \
  --epochs 10 \
  --batch-size 32 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2

In [ ]:
from thresholds import optimize_thresholds, save_thresholds, apply_thresholds
from metrics import alta_score
import pandas as pd

valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/transformer_base/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/transformer_base/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print("BASE MODEL CALIBRATED:", scores, final)

In [ ]:
!python transformer_multitask.py \
  --train train.csv \
  --valid valid.csv \
  --out artifacts/transformer_large \
  --model microsoft/deberta-v3-large \
  --epochs 8 \
  --batch-size 4 \
  --max-length 192 \
  --n-folds 3 \
  --patience 2 \
  --grad-accum 8

In [ ]:
valid = pd.read_csv('valid.csv')
probs = pd.read_csv('artifacts/transformer_large/valid_probabilities.csv')
th = optimize_thresholds(valid, probs)
save_thresholds(th, 'artifacts/transformer_large/thresholds.json')
pred = apply_thresholds(probs, th)
scores, final = alta_score(valid, pred)
print("LARGE MODEL CALIBRATED:", scores, final)

In [ ]:
%%writefile sweep_blend.py
import sys
import pandas as pd
from thresholds import optimize_thresholds, apply_thresholds
from metrics import alta_score

prob_b_path = sys.argv[1] if len(sys.argv) > 1 else "artifacts/transformer_large/valid_probabilities.csv"

valid = pd.read_csv("valid.csv")
a = pd.read_csv("artifacts/classical/valid_probabilities.csv")
b = pd.read_csv(prob_b_path)

cols = ["source", "variety", "text"]
assert a[cols].equals(b[cols]) and a[cols].equals(valid[cols]), "row mismatch"

results = []
for wa in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    wb = 1 - wa
    p = valid[cols].copy()
    p["p_sentiment"] = wa * a["p_sentiment"] + wb * b["p_sentiment"]
    p["p_sarcasm"] = wa * a["p_sarcasm"] + wb * b["p_sarcasm"]
    th = optimize_thresholds(valid, p)
    pred = apply_thresholds(p, th)
    scores, final = alta_score(valid, pred)
    results.append({"weight_classical": wa, "weight_transformer": wb, "alta_score": final, **scores})
    print(f"wa={wa:.1f} wb={wb:.1f} -> ALTA={final:.4f}  {scores}")

df = pd.DataFrame(results).sort_values("alta_score", ascending=False)
print(f"\nBest blend for {prob_b_path}:")
print(df.iloc[0])

In [ ]:
!python sweep_blend.py artifacts/transformer_large/valid_probabilities.csv

In [ ]:
!python sweep_blend.py artifacts/transformer_base/valid_probabilities.csv

In [ ]:
!cd /kaggle/working/alta2026_pipeline && zip -r all_artifacts.zip artifacts/